In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# =========================
# 1. Imports & Config
# =========================
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import joblib
from pathlib import Path

# -------------------------
# CONFIG (EDIT PATHS ONLY)
# -------------------------
TRAIN_NPZ_PATH = "/kaggle/input/dino3embeddings/train_embeddings.npz"
TEST_NPZ_PATH  = "/kaggle/input/dino3embeddings/test_embeddings.npz"
OUTPUT_DIR     = "output_models"

N_FOLDS = 5
BATCH_SIZE = 128
EPOCHS = 200
LR = 1e-3
SEED = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)


# =========================
# 2. Load Embeddings
# =========================
train_data = np.load(TRAIN_NPZ_PATH)
X = train_data["X"].astype(np.float32)   # shape: (N, D)
y = train_data["y"].astype(np.float32)   # shape: (N, 15)

print("Train embeddings:", X.shape)
print("Train targets:", y.shape)


# =========================
# 3. Dataset
# =========================
class EmbeddingDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.from_numpy(X)
        self.y = None if y is None else torch.from_numpy(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is None:
            return self.X[idx]
        return self.X[idx], self.y[idx]


# =========================
# 4. Model (Fast & Strong)
# =========================

class MLPRegressor(nn.Module):
    def __init__(self, in_dim, dropout=0.2):
        super().__init__()


        # shared trunk (same as before conceptually)
        self.shared = nn.Sequential(
            nn.Linear(in_dim, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout),

            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(256),
        )

        # 15 independent heads
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(256, 64),
                nn.ReLU(inplace=True),
                nn.Linear(64, 1)
            )
            for _ in range(15)
        ])

    def forward(self, x):
        
        
        feat = self.shared(x)

        outputs = []
        for head in self.heads:
            outputs.append(head(feat))

        # (B, 15)
        return torch.cat(outputs, dim=1)




# =========================
# 5. Train / Validate
# =========================
criterion = nn.L1Loss()  # MAE (matches competition)

def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader):
    model.eval()
    preds_all = []
    y_all = []

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        preds = model(xb).cpu().numpy()
        preds_all.append(preds)
        y_all.append(yb.numpy())

    preds_all = np.vstack(preds_all)
    y_all = np.vstack(y_all)

    return mean_absolute_error(y_all, preds_all)


# =========================
# 6. K-Fold Training
# =========================
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_preds = np.zeros_like(y)
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(X)):
    print(f"\n========== Fold {fold} ==========")

    X_tr, X_va = X[tr_idx], X[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    # ---- Target Scaling (per fold)
    scaler = StandardScaler()
    y_tr_scaled = scaler.fit_transform(y_tr)
    y_va_scaled = scaler.transform(y_va)


    # Save scaler
    joblib.dump(scaler, f"{OUTPUT_DIR}/scaler_fold{fold}.pkl")

    # ---- Datasets
    tr_ds = EmbeddingDataset(X_tr, y_tr_scaled)
    va_ds = EmbeddingDataset(X_va, y_va_scaled)

    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False)

    # ---- Model
    model = MLPRegressor(in_dim=X.shape[1]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

    best_mae = np.inf

    for epoch in range(EPOCHS):
        tr_loss = train_one_epoch(model, tr_loader, optimizer)
        va_mae = validate(model, va_loader)

        print(f"Epoch {epoch+1:02d} | Train L1: {tr_loss:.4f} | Val MAE: {va_mae:.4f}")

        if va_mae < best_mae:
            best_mae = va_mae
            torch.save(
                model.state_dict(),
                f"{OUTPUT_DIR}/mlp_fold{fold}.pth"
            )

    print(f"Best Fold {fold} MAE: {best_mae:.4f}")
    fold_scores.append(best_mae)

print("\nCV MAE:", np.mean(fold_scores))

Train embeddings: (938, 2048)
Train targets: (938, 15)

========== Fold 0 ==========
Epoch 01 | Train L1: 0.5002 | Val MAE: 0.7338
Epoch 02 | Train L1: 0.3414 | Val MAE: 0.6448
Epoch 03 | Train L1: 0.2793 | Val MAE: 0.5344
Epoch 04 | Train L1: 0.2546 | Val MAE: 0.4524
Epoch 05 | Train L1: 0.2244 | Val MAE: 0.3447
Epoch 06 | Train L1: 0.2147 | Val MAE: 0.2919
Epoch 07 | Train L1: 0.2110 | Val MAE: 0.2695
Epoch 08 | Train L1: 0.2021 | Val MAE: 0.2323
Epoch 09 | Train L1: 0.2011 | Val MAE: 0.2317
Epoch 10 | Train L1: 0.1839 | Val MAE: 0.2012
Epoch 11 | Train L1: 0.1806 | Val MAE: 0.1934
Epoch 12 | Train L1: 0.1747 | Val MAE: 0.2079
Epoch 13 | Train L1: 0.1740 | Val MAE: 0.1972
Epoch 14 | Train L1: 0.1696 | Val MAE: 0.1956
Epoch 15 | Train L1: 0.1685 | Val MAE: 0.1816
Epoch 16 | Train L1: 0.1699 | Val MAE: 0.2013
Epoch 17 | Train L1: 0.1614 | Val MAE: 0.1981
Epoch 18 | Train L1: 0.1595 | Val MAE: 0.2005
Epoch 19 | Train L1: 0.1542 | Val MAE: 0.1721
Epoch 20 | Train L1: 0.1517 | Val MAE: 0.

In [3]:
# =========================
# 1. Imports & Config
# =========================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import joblib
from pathlib import Path

# -------------------------
# EDIT THESE PATHS
# -------------------------
TEST_NPZ_PATH = "/kaggle/input/dino3embeddings/test_embeddings.npz"
SAMPLE_SUB_PATH = "/kaggle/input/dataset-africarice/Unido_AfricaRice_Challenge/SampleSubmission.csv"
MODEL_DIR = "output_models"
SUBMISSION_PATH = "submission.csv"

N_FOLDS = 5
BATCH_SIZE = 256
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

target_columns = [
    'Count', 'Broken_Count', 'Long_Count', 'Medium_Count',
    'Black_Count', 'Chalky_Count', 'Red_Count', 'Yellow_Count',
    'Green_Count', 'WK_Length_Average', 'WK_Width_Average',
    'WK_LW_Ratio_Average', 'Average_L', 'Average_a', 'Average_b'
]


# =========================
# 2. Dataset
# =========================
class EmbeddingDataset(Dataset):
    def __init__(self, X):
        self.X = torch.from_numpy(X)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx]



# =========================
# 4. Load Test Embeddings
# =========================
test_data = np.load(TEST_NPZ_PATH)
X_test = test_data["X"].astype(np.float32)

print("Test embeddings shape:", X_test.shape)

test_loader = DataLoader(
    EmbeddingDataset(X_test),
    batch_size=BATCH_SIZE,
    shuffle=False
)


# =========================
# 5. K-Fold Inference
# =========================
test_preds = np.zeros((len(X_test), 15), dtype=np.float32)

for fold in range(N_FOLDS):
    print(f"Inference with fold {fold}")

    # Load scaler
    scaler = joblib.load(f"{MODEL_DIR}/scaler_fold{fold}.pkl")

    # Load model
    model = MLPRegressor(in_dim=X_test.shape[1]).to(DEVICE)
    model.load_state_dict(
        torch.load(f"{MODEL_DIR}/mlp_fold{fold}.pth", map_location=DEVICE)
    )
    model.eval()

    fold_preds = []

    with torch.no_grad():
        for xb in test_loader:
            xb = xb.to(DEVICE)
            preds_scaled = model(xb).cpu().numpy()
            preds = scaler.inverse_transform(preds_scaled)
            fold_preds.append(preds)

    fold_preds = np.vstack(fold_preds)
    test_preds += fold_preds / N_FOLDS


# =========================
# 6. Build Submission
# =========================
sub_df = pd.read_csv(SAMPLE_SUB_PATH)

assert len(sub_df) == test_preds.shape[0], \
    "Mismatch between submission rows and predictions"

sub_df[target_columns] = test_preds


# =========================
# 7. Post-processing
# =========================
# Clamp negatives
sub_df[target_columns] = sub_df[target_columns].clip(lower=0)

# Round count-based targets
count_cols = [
    'Count', 'Broken_Count', 'Long_Count', 'Medium_Count',
    'Black_Count', 'Chalky_Count', 'Red_Count',
    'Yellow_Count', 'Green_Count'
]
sub_df[count_cols] = sub_df[count_cols].round()


# =========================
# 8. Save Submission
# =========================
sub_df.to_csv(SUBMISSION_PATH, index=False)

print("Submission saved to:", SUBMISSION_PATH)
print(sub_df.head())
print("Submission shape:", sub_df.shape)


Test embeddings shape: (403, 2048)
Inference with fold 0
Inference with fold 1
Inference with fold 2
Inference with fold 3
Inference with fold 4
Submission saved to: submission.csv
          ID   Count  Broken_Count  Long_Count  Medium_Count  Black_Count  \
0  ID_0537FB  1366.0         473.0       722.0           0.0        228.0   
1  ID_903CE9  1451.0         509.0       650.0           3.0        621.0   
2  ID_37F0E8   870.0         187.0        95.0           0.0        836.0   
3  ID_26DD0D  1536.0         347.0       991.0           1.0        468.0   
4  ID_3B8A61  1315.0         460.0       350.0           0.0       1325.0   

   Chalky_Count  Red_Count  Yellow_Count  Green_Count  WK_Length_Average  \
0        1024.0       20.0          70.0          0.0           7.214961   
1         443.0       24.0         345.0          2.0           7.342972   
2           0.0       54.0           1.0          0.0           9.974257   
3         600.0       60.0         404.0          1.

In [4]:
import pandas as pd
import numpy as np

# =========================
# Paths
# =========================
SUB1_PATH = "submission.csv"   # weight 0.6
SUB2_PATH ="/kaggle/input/zindiafricanrice/submission_convnext.csv"   # weight 0.4
OUT_PATH  = "submission_ensemble.csv"

W1 = 0.75
W2 = 0.25

# =========================
# Load submissions
# =========================
sub1 = pd.read_csv(SUB1_PATH)
sub2 = pd.read_csv(SUB2_PATH)

# =========================
# Safety checks
# =========================
assert sub1.shape == sub2.shape, "Submission shapes do not match"
assert (sub1["ID"].values == sub2["ID"].values).all(), "IDs are not aligned"

TARGET_COLS = [c for c in sub1.columns if c != "ID"]

# =========================
# Weighted ensemble
# =========================
ensemble = sub1.copy()

ensemble[TARGET_COLS] = (
    W1 * sub1[TARGET_COLS].values +
    W2 * sub2[TARGET_COLS].values
)

# =========================
# Post-processing (same as training)
# =========================
ensemble[TARGET_COLS] = ensemble[TARGET_COLS].clip(lower=0)

# Round count targets (first 9)
count_cols = TARGET_COLS[:9]
ensemble[count_cols] = ensemble[count_cols].round()

# =========================
# Save
# =========================
ensemble.to_csv(OUT_PATH, index=False)

print(f"✓ Ensemble saved to {OUT_PATH}")
print(ensemble.head())


✓ Ensemble saved to submission_ensemble.csv
          ID   Count  Broken_Count  Long_Count  Medium_Count  Black_Count  \
0  ID_0537FB  1372.0         457.0       719.0           0.0        237.0   
1  ID_903CE9  1460.0         498.0       664.0           2.0        616.0   
2  ID_37F0E8   884.0         192.0       103.0           0.0        842.0   
3  ID_26DD0D  1518.0         332.0       981.0           1.0        448.0   
4  ID_3B8A61  1307.0         456.0       346.0           0.0       1314.0   

   Chalky_Count  Red_Count  Yellow_Count  Green_Count  WK_Length_Average  \
0        1020.0       18.0          64.0          2.0           7.200408   
1         460.0       26.0         332.0          3.0           7.321182   
2           2.0       49.0           2.0          1.0           9.850854   
3         599.0       57.0         422.0          2.0           7.482142   
4           2.0       17.0           3.0          1.0           8.851357   

   WK_Width_Average  WK_LW_Ratio_Ave